In [5]:
import numpy as np
import matplotlib.pyplot as plt 
from PIL import Image 
import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint

In [ ]:
#　画像読み込み
#.convert("L")で開いた画像を別の色の形式に変換する命令。"L"はグレースケールを意味する。
img= Image.open("Images/Peppers.png").convert("L")

#　解像度
img=img.resize((256,256))

# numpy化。imgの画像の各画素値0~256の値に変換して配列にする。
# .astype(np.float32)で便宜上データの種類を32ビット浮動小数点数に変換（0→0.0）
img_array=np.array(img).astype(np.float32)

# 画素値（0～255）を0～1へ変換
img_array=img_array/255.0

In [7]:
#　画像の縦横が何マスか
height,width=img_array.shape

x=np.linspace(1,-1,width)
y=np.linspace(1,-1,height)

X,Y=np.meshgrid(x,y)

In [8]:
print(X)

[[ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 ...
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]]


In [9]:
print(Y)

[[ 1.          1.          1.         ...  1.          1.
   1.        ]
 [ 0.99215686  0.99215686  0.99215686 ...  0.99215686  0.99215686
   0.99215686]
 [ 0.98431373  0.98431373  0.98431373 ...  0.98431373  0.98431373
   0.98431373]
 ...
 [-0.98431373 -0.98431373 -0.98431373 ... -0.98431373 -0.98431373
  -0.98431373]
 [-0.99215686 -0.99215686 -0.99215686 ... -0.99215686 -0.99215686
  -0.99215686]
 [-1.         -1.         -1.         ... -1.         -1.
  -1.        ]]


In [ ]:
# 欠損作成
# np.ones() は、指定されたサイズで「中身がすべて 1 のデータ」を作る関数
# dtype=boolで数字の１ではなくBoolean型（True/False）に変換。１の場合はTrue
# 後の欠損部分をくりぬく処理で便利
mask=np.ones(img_array.shape,dtype=bool)


np.random.seed(42)
num_missing=int(height*width*0.05)

#　↓この条件をもとに欠損させる場所を指定
indices= np.random.choice(
    height*width,
    num_missing,
    replace=False #「重複無し」の意味。一度選んだピクセルの位置は二度とえらばない→正確にかぶりなく5％分の場所を確保できる
)

# リスト内包表記
# 5%分のランダムな通し番号（indices）から、数字を1つずつ順番に取り出して idx（インデックス）という変数に入れています。この処理をデータの数だけぐるぐる繰り返します。
# idx // width（スラッシュ2つ）：idx を画像の横幅（width）で割った「商（割り算の答えの整数部分）」です。これが「縦（上から何行目か）」を表します。
# idx % width（パーセント）：idx を画像の横幅（width）で割った「余り」です。これが「横（左から何列目か）」を表します。
"""
例えば、横幅（width）が 100 の画像があるとします。
通し番号（idx）が 253 番だった場合、これを縦と横に直すとどうなるでしょうか？
・縦： 253 // 100 = 2 （上から3行目 ※0から数えるため）
・横： 253 % 100 = 53 （左から54列目）
つまり、253番という1列の背番号が、(2, 53) という「2行目の53列目」という画像の具体的なマス目の位置（座標）に見事変換されます。
"""
missing_points=[
    (idx//width,idx % width)
    for idx in indices
]

# 欠損化
for i,j in missing_points:
    mask[i,j]=False # 欠損にさせたい座標(i,j)だけをFalseにする


# 元の画像データを直接書き換えるのではなく、あえて.copyを使って複製をつくっている。
#こうすることで万が一この後の実験で失敗しても元のきれいな画像データを汚さずに済む（プログラミングにおいて重要な安全対策）
img_missing= img_array.copy()

#現在は欠損座標がFalseそれ以外の観測データがTrueとなっているが、これを「~」（プログラミングにおいて反転を意味する）を使って欠損座標をTrueに、観測データをFalseする。
#理由はNumpyの仕様で「配列[条件]」と書いたとき、「Trueの場所のデータを変更する」という決まりになっているから
#maskは上で定義したもの。ture/falseで埋められた元画像と同じサイズの地図のようなモノ
#maskという条件を元画像の画素値が入ったデータの上にピタッと張り付けてTrueのところだけnanにしているイメージ
img_missing[~mask]=np.nan

In [15]:
print(img_missing)

[[0.         0.         0.         ...        nan 0.         0.        ]
 [0.         0.45490196 0.43529412 ... 0.7137255  0.6745098  0.8156863 ]
 [0.         0.4862745  0.44313726 ... 0.7176471  0.6901961  0.827451  ]
 ...
 [0.         0.5019608  0.4        ... 0.76862746 0.7607843         nan]
 [0.         0.5294118  0.43529412 ... 0.7921569  0.74509805 0.9529412 ]
 [0.         0.45882353 0.4392157  ... 0.79607844 0.7490196  0.9098039 ]]


In [16]:
#学習データ作成
# .flattenで二次元の四角い形を崩し、一列の長い数字のリストにする
"""
例えば、縦が 2 マス、横が 3 マス（合計6マス）の小さな画像があるとします。
もともとの X と Y は、以下のような2次元の形をしています。
X (横の座標): [[0, 1, 2], [0, 1, 2]]
Y (縦の座標): [[0, 0, 0], [1, 1, 1]]
これを .flatten() すると、1次元の長いデータになります。
X.flatten() \(\rightarrow \) [0, 1, 2, 0, 1, 2]
Y.flatten() \(\rightarrow \) [0, 0, 0, 1, 1, 1]
axis=1 の意味: 「横方向（列方向）に合体させる」という指示です。
最後に、np.stack(..., axis=1) で左右に合体させると、coords の中身は以下のようになります。
# coordsの中身（2列のリストになる）
[
  [0, 0],  # 1マス目の座標 (横0, 縦0)
  [1, 0],  # 2マス目の座標 (横1, 縦0)
  [2, 0],  # 3マス目の座標 (横2, 縦0)
  [0, 1],  # 4マス目の座標 (横0, 縦1)
  [1, 1],  # 5マス目の座標 (横1, 縦1)
  [2, 1]   # 6マス目の座標 (横2, 縦1)
]

「コンピューターが計算しやすい座標の一覧表」が完成
"""
coords=np.stack(
    [X.flatten(),Y.flatten()],
    axis=1
)

#全画素の色の数値の一次元リストを作る処理
targets=img_array.flatten()

#データが無事か欠損しているかの一次元地図を作る処理
mask_flat=mask.flatten()

#AIに学習させるための「問題（無事な座標リスト）」を作る処理
coords_train=coords[mask_flat]

#AIに学習させるための「答え（無事な画素値リスト）」を作る処理
targets_train=targets[mask_flat]

#Numpyで作った座標データをPyTorchライブラリで計算できる専用の形式（テンソル）に変換する処理
coords_tensor=torch.tensor(
    coords_train,
    dtype=torch.float32# AIのモデルはこのfloat32（浮動小数点数）という型でデータを入力されることを基本ルールとしている。もし整数のまま入力してしまうとエラーになる
)

targets_tensor=torch.tensor(
    targets_train,
    dtype=torch.float32
).view(-1,1)# データの形を縦一列（二次元の縦ベクトル）に変形する
"""
解説: ここが一番のポイントです。
変換する前の targets_train は、ただ数字が横1列に並んだ「1次元のリスト」のような形（例: [255, 120, 0, ...]）をしています。
しかし、PyTorchのAIモデル（ニューラルネットワーク）で損失（誤差）を計算するとき、
問題データと答えデータの次元（データの外枠の形）が一致していないとエラーが起きてしまいます。
そこで .view(-1, 1) を使って、「横幅は『1』マス、縦の長さ（-1）はデータ数に合わせて自動計算」という指定をします。
これにより、データが「縦1列のきれいな表（2次元）」に変形されます。
"""

'\n解説: ここが一番のポイントです。\n変換する前の targets_train は、ただ数字が横1列に並んだ「1次元のリスト」のような形（例: [255, 120, 0, ...]）をしています。\nしかし、PyTorchのAIモデル（ニューラルネットワーク）で損失（誤差）を計算するとき、\n問題データと答えデータの次元（データの外枠の形）が一致していないとエラーが起きてしまいます。\nそこで .view(-1, 1) を使って、「横幅は『1』マス、縦の長さ（-1）はデータ数に合わせて自動計算」という指定をします。\nこれにより、データが「縦1列のきれいな表（2次元）」に変形されます。\n'

In [22]:
targets_train

array([0.        , 0.        , 0.        , ..., 0.79607844, 0.7490196 ,
       0.9098039 ], shape=(62260,), dtype=float32)

In [20]:
targets_tensor#実行結果がtargets_trainよりも桁が省略されているように見えるが、これは画面表示用に省略されており、内部ではしっかり元の数値が保持されている

tensor([[0.0000],
        [0.0000],
        [0.0000],
        ...,
        [0.7961],
        [0.7490],
        [0.9098]])

In [ ]:
#ODE Function
class ODEFunc(nn.Module):
    def __init__(self):
        super().__init__()# PyTorchが持つAIの基本機能をすべてこのクラスに引き継ぐ（継承する）ための呪文。お約束の一文
        self.net=nn.Sequential( #4つの層を順番につないだ一本道のパイプラインを作っている 

            nn.Linear(2,128),
            nn.Tanh(),

            nn.Linear(128,128),
            nn.Tanh(),

            nn.Linear(128,128),
            nn.Tanh(),

            nn.Linear(128,2)
        )
    def forward(self,t,h):# データがこのネットワークを通り抜けるときの実際の計算ルールを書いている

        return self.net(h)
    
"""
このモデルの面白いところは、出口が 2 になっている点です。
「あれ？正解データ（targets_tensor）は色の数値の『1つ』のはずでは？」と思いますよね。
Neural ODEでは、このネットワークは「色そのもの」を直接出力するのではなく、
「画像を綺麗に修復していくための、データの動く方向や勢い（ベクトル）」 を出力しています。
この出力された「勢い」を使って、別のシステム（ODEソルバー）が少しずつデータを変化させ、最終的に綺麗な1つの色へと導いていきます。
"""